In [1]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Row
spark = (
    SparkSession.builder
    .appName("retrainmodel")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/16 09:03:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/16 09:03:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/16 09:03:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/16 09:03:16 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/01/16 09:03:16 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
import happybase

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')

rows = []
for key, data in table.scan():
    if b'#1m' in key:
        key_str = key.decode()
        try:
            symbol, ts_str, interval = key_str.split('#')
            timestamp = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
        except Exception as e:
            # jeśli key nie pasuje do formatu, pomiń
            continue

        row_dict = {
            'symbol': symbol,
            'timestamp': timestamp,
            'interval': interval
        }

        for col, val in data.items():
            col_name = col.decode()
            try:
                row_dict[col_name] = float(val.decode())
            except ValueError:
                row_dict[col_name] = val.decode()

        rows.append(Row(**row_dict))

df = spark.createDataFrame(rows)

df.show(truncate=False)

26/01/16 09:06:28 WARN TaskSetManager: Stage 8 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
[Stage 8:>                                                          (0 + 1) / 1]

+------+-------------------+--------+----------+---------+---------+---------+
|symbol|timestamp          |interval|ohlc:close|ohlc:high|ohlc:low |ohlc:open|
+------+-------------------+--------+----------+---------+---------+---------+
|BTC   |2025-11-06 21:37:00|1m      |100937.93 |100937.93|100937.93|100937.93|
|BTC   |2025-11-06 21:38:00|1m      |100946.74 |100946.74|100922.79|100922.79|
|BTC   |2025-11-06 21:39:00|1m      |100932.99 |100932.99|100918.83|100930.14|
|BTC   |2025-11-06 21:40:00|1m      |100911.99 |100911.99|100898.96|100898.96|
|BTC   |2025-11-06 21:41:00|1m      |100951.99 |100951.99|100924.54|100924.55|
|BTC   |2025-11-06 21:42:00|1m      |101137.41 |101137.41|100976.02|100976.02|
|BTC   |2025-11-06 21:43:00|1m      |101146.47 |101146.47|101120.14|101137.4 |
|BTC   |2025-11-06 21:44:00|1m      |101016.37 |101090.25|101016.37|101090.25|
|BTC   |2025-11-06 21:45:00|1m      |100963.82 |101000.01|100963.82|101000.01|
|BTC   |2025-11-06 21:46:00|1m      |100942.81 |1009

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col

symbols_to_keep = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]
df_filtered = df.filter(col("symbol").isin(symbols_to_keep))

window = Window.partitionBy("symbol").orderBy("timestamp")

df_filtered = df_filtered.withColumn("next_close", lead("ohlc:close", 1).over(window))

In [5]:
from pyspark.sql.functions import first, col

symbols = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]

# Pivot ceny
df_close = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("ohlc:close"))

# Pivot next_close z nowymi nazwami
df_next = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("next_close"))

# Zmieniamy nazwy kolumn next_close przed joinem
for sym in symbols:
    if sym in df_next.columns:
        df_next = df_next.withColumnRenamed(sym, f"{sym}_next_close")

# Join po timestamp
df_pivot = df_close.join(df_next, on="timestamp", how="inner")

# Konwersja kolumn na double (oprócz timestamp)
for col_name in df_pivot.columns:
    if col_name != "timestamp":
        df_pivot = df_pivot.withColumn(col_name, col(col_name).cast("double"))
        
feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"
df_pivot = df_pivot.dropna(subset=feature_cols + [label_col])
# df_pivot.show(5)


In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"

df_ml = df_pivot.dropna(subset=feature_cols + [label_col])

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df_ml)\
    .select("timestamp", "features", col(label_col).cast("double").alias("label"))

from pyspark.sql.functions import to_timestamp, col

df_ml = df_ml.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

In [9]:
model_info = spark.read.option("header", True).csv("hdfs://namenode:8020/models/model_registry_csv/")

In [7]:
model_info.show(5, truncate=False)

+----------+--------------------------------------------+-----------------+------------------+-----------------+------------------------+
|model_name|model_path                                  |mse              |r2                |rmse             |timestamp               |
+----------+--------------------------------------------+-----------------+------------------+-----------------+------------------------+
|GBTR_BTC_2|hdfs://namenode:8020/models/btc_model_GBTR_2|597375.0407382783|0.4757359113231875|772.9004080334531|2026-01-14T17:41:04.535Z|
+----------+--------------------------------------------+-----------------+------------------+-----------------+------------------------+



In [10]:
from pyspark.sql.functions import col, max

# Najpierw znajdź maksymalny timestamp
latest_ts = model_info.select(max("timestamp")).first()[0]

# Filtruj wiersz z tym timestampem
latest_row = model_info.filter(col("timestamp") == latest_ts).first()

# Pobierz wartości
model_path = latest_row["model_path"]
timestamp = latest_row["timestamp"]



Najświeższy model_path: hdfs://namenode:8020/models/btc_model_GBTR_2
Timestamp: 2026-01-14T17:41:04.535Z


In [12]:
from pyspark.ml.regression import GBTRegressionModel

# model = GBTRegressionModel.load(model_path)
# hdfs://namenode:8020/models/btc_model_GBTR_2
model = GBTRegressionModel.load('hdfs://namenode:8020/models/btc_model_GBTR')

In [13]:
predictions = model.transform(train_df.limit(5))

In [14]:
predictions.show()

26/01/16 09:15:00 WARN TaskSetManager: Stage 44 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/16 09:15:02 WARN TaskSetManager: Stage 45 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------------------+--------------------+--------+-----------------+
|          timestamp|            features|   label|       prediction|
+-------------------+--------------------+--------+-----------------+
|2025-11-15 11:41:00|[22900.58984375,6...|95811.32|95473.96964131563|
|2025-11-15 11:43:00|[22900.58984375,6...|95807.61|95473.96964131563|
|2025-11-15 11:45:00|[22900.58984375,6...|95722.02|95410.89175486895|
|2025-11-15 11:46:00|[22900.58984375,6...|95742.65|95410.89175486895|
|2025-11-15 11:47:00|[22900.58984375,6...|95684.74|95410.89175486895|
+-------------------+--------------------+--------+-----------------+



In [7]:
from pyspark.sql.functions import col, current_timestamp, date_sub


cutoff = date_sub(current_timestamp(), 2)

train_df = df_ml.filter(col("timestamp") < cutoff)
test_df = df_ml.filter(col("timestamp") >= cutoff)

In [9]:
train_df.show(3,truncate=False)

26/01/16 09:08:01 WARN TaskSetManager: Stage 18 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/16 09:08:02 WARN TaskSetManager: Stage 19 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------------------+---------------------------------------------------------------+--------+
|timestamp          |features                                                       |label   |
+-------------------+---------------------------------------------------------------+--------+
|2025-11-15 11:41:00|[22900.58984375,6734.10986328125,47147.48046875,140.7,3161.97] |95811.32|
|2025-11-15 11:43:00|[22900.58984375,6734.10986328125,47147.48046875,140.59,3161.35]|95807.61|
|2025-11-15 11:45:00|[22900.58984375,6734.10986328125,47147.48046875,140.14,3154.02]|95722.02|
+-------------------+---------------------------------------------------------------+--------+
only showing top 3 rows



In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=200,        # liczba drzew
    maxDepth=8,         # głębokość drzew
    stepSize=0.05,      # learning rate
    subsamplingRate=0.8,
    seed=42
)

model = gbt.fit(train_df)

In [ ]:
predictions = model.transform(test_df)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mse"
)

mse = evaluator.evaluate(predictions)
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)

In [ ]:
predictions.write.mode("append").saveAsTable("cryptopredictions.btc_predictions_GBTR_2")

In [ ]:
model.write().overwrite().save("hdfs://namenode:8020/models/btc_model_GBTR_2")

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit

model_path = "hdfs://namenode:8020/models/btc_model_GBTR_2"
df_model_info = spark.createDataFrame([{
    "model_name": "GBTR_BTC_2",
    "model_path": model_path,
    "rmse": rmse,
    "mse": mse,
    "r2": r2
}])
df_model_info = df_model_info.withColumn("timestamp", current_timestamp())
df_model_info.show(truncate=False)


In [ ]:
df_model_info.write.mode("append").option("header", True).csv(
    "hdfs://namenode:8020/models/model_registry_csv/"
)

In [ ]:
spark.stop()